# Лабораторна робота №4. Регресійні моделі: глибоке дослідження

**Датасет:** Car Price Prediction Multiple Linear Regression  
https://www.kaggle.com/datasets/hellbuoy/car-price-prediction?select=CarPrice_Assignment.csv

**Мета роботи:** виконати підготовку даних, побудувати та порівняти регресійні моделі для прогнозування ціни автомобіля, оцінити їх якість і проаналізувати отримані результати.

> У шаблоні наведено лише умови завдань. Реалізацію необхідно виконати самостійно у порожніх комірках.
**Виконав:** Харченко Артьом, група ФІТ 3-10

## 1. Завантаження та первинний аналіз даних

Завантажити датасет та виконати його первинний огляд:
- визначити кількість об'єктів і ознак;
- переглянути назви стовпців;
- визначити типи даних;
- визначити цільову змінну;
- вивести основні статистичні характеристики даних.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

path = kagglehub.dataset_download("hellbuoy/car-price-prediction")
df = pd.read_csv(os.path.join(path, "CarPrice_Assignment.csv"))

print("Об'єктів:", df.shape[0])
print("Ознак:", df.shape[1])
print("Стовпці:", df.columns.tolist())
print()
df.info()

print()
print("Цільова змінна: price (ціна автомобіля)")
df.describe()

**Висновок:** у датасеті 205 автомобілів і 26 ознак. Цільова змінна price, це ціна машини, вона від 5118 до 45400, в середньому близько 13277. Числові ознаки описують розміри, вагу і двигун, а текстові це тип палива, кузов, привід, тип двигуна і т.д.

## 2. Перевірка якості даних

Перевірити:
- наявність пропущених значень;
- наявність повних дублікатів;
- кількість унікальних значень в ознаках.

Під час аналізу врахувати, що `car_ID` є унікальним ідентифікатором автомобіля і сам по собі не повинен використовуватися для визначення дублікатів за змістовими характеристиками.


In [ ]:
print("Пропусків у кожному стовпці:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Всього пропусків:", df.isna().sum().sum())
print()

# car_ID у всіх різний, тому дублікати шукаємо без нього
print("Повних дублікатів (без car_ID):", df.drop(columns="car_ID").duplicated().sum())
print()
print("Унікальних значень:")
print(df.nunique())

In [ ]:
# у CarName 147 різних значень (марка + модель), для 205 машин це забагато
# беремо тільки марку (перше слово) і виправляємо помилки в назвах
df["brand"] = df["CarName"].str.split().str[0].str.lower()
print("До виправлення:", sorted(df["brand"].unique()))

df["brand"] = df["brand"].replace({
    "maxda": "mazda",
    "porcshce": "porsche",
    "toyouta": "toyota",
    "vokswagen": "volkswagen",
    "vw": "volkswagen",
})
print("Після виправлення:", sorted(df["brand"].unique()))
print("Марок:", df["brand"].nunique())

df = df.drop(columns="CarName")

**Висновок:** пропусків і повних дублікатів немає. У CarName 147 різних значень, тобто назва моделі майже у кожної машини своя, і як ознака вона нічого не дасть. Тому з неї взяли тільки марку і виправили помилки в назвах (maxda, porcshce, toyouta, vokswagen, vw), вийшло 22 марки. Сам стовпець CarName видалили.

## 3. Робота з car_ID

Зберегти `car_ID` окремо для подальшої ідентифікації автомобілів у результатах прогнозування.  
Не використовувати `car_ID` як ознаку під час навчання моделей.


In [ ]:
# зберігаємо car_ID окремо, щоб потім знати, для якої машини який прогноз
car_id = df["car_ID"]
print(car_id.head())

## 4. Дослідження розподілів числових ознак

Дослідити розподіли числових ознак:
- побудувати відповідні графіки;
- обчислити коефіцієнти асиметрії;
- визначити ознаки з найбільш вираженою асиметрією;
- коротко прокоментувати отримані результати.


In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop("car_ID").tolist()

fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=20)
    axes[i].set_title(col)
for j in range(len(num_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

skew = df[num_cols].skew().sort_values(ascending=False)
print("Коефіцієнти асиметрії:")
print(skew.round(3))

**Висновок:** найсильніша асиметрія у compressionratio (2.61), enginesize (1.95), price (1.78) і horsepower (1.41). Всі вони скошені вправо: більшість машин звичайні, і є кілька дуже потужних або дорогих. У compressionratio є окрема група значень 21-23, це дизельні машини. stroke трохи скошена вліво (-0.69).

## 5. Формування train/test вибірок

Відокремити цільову змінну `price` від ознак та розділити дані на:
- навчальну вибірку — 80%;
- тестову вибірку — 20%.

Забезпечити відтворюваність поділу.  
Окремо зберегти `car_ID` для об'єктів, які потрапили до train і test.

**Важливо:** усі подальші параметри передобробки даних визначати тільки за навчальною вибіркою.


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["car_ID", "price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# car_ID для машин з train і test
id_train = car_id.loc[X_train.index]
id_test = car_id.loc[X_test.index]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 6. Аналіз та обробка викидів

Для неперервних числових ознак навчальної вибірки:
- визначити потенційні викиди методом IQR;
- вивести кількість потенційних викидів для кожної ознаки;
- обґрунтувати спосіб їх обробки.

Для невеликого набору даних не видаляти автоматично всі рядки з викидами. Використати обмеження крайніх значень межами, визначеними за навчальною вибіркою.

Цільову змінну `price` не використовувати для визначення меж викидів.  
До тестової вибірки застосувати ті самі межі, які були отримані за train.


In [ ]:
# неперервні числові ознаки (symboling дискретна, price - ціль, їх не беремо)
cont_cols = ["wheelbase", "carlength", "carwidth", "carheight", "curbweight", "enginesize",
             "boreratio", "stroke", "compressionratio", "horsepower", "peakrpm", "citympg", "highwaympg"]

# межі рахуємо тільки по train
q1 = X_train[cont_cols].quantile(0.25)
q3 = X_train[cont_cols].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = ((X_train[cont_cols] < lower) | (X_train[cont_cols] > upper)).sum()
print("Кількість потенційних викидів у train:")
print(outliers)

In [ ]:
# рядки не видаляємо, бо даних мало, а просто обрізаємо значення по межах з train
X_train = X_train.copy()
X_test = X_test.copy()
X_train[cont_cols] = X_train[cont_cols].clip(lower, upper, axis=1)
X_test[cont_cols] = X_test[cont_cols].clip(lower, upper, axis=1)

print("Викидів після обрізання:", ((X_train[cont_cols] < lower) | (X_train[cont_cols] > upper)).sum().sum())

**Висновок:** найбільше потенційних викидів у compressionratio (20, це якраз дизелі) і stroke (16). Рядки з викидами не видаляли, бо в train всього 164 машини і ми б втратили багато даних, до того ж це реальні машини, а не помилки. Тому значення обрізали по межах IQR, які порахували на train, і ці ж межі застосували до test. Ціну price для меж не використовували.

## 7. Аналіз кореляцій та відбір ознак

За навчальною вибіркою:
- побудувати матрицю кореляцій числових ознак;
- знайти пари ознак із сильною кореляцією, для яких `|r| > 0.85`;
- для кожної такої пари залишити одну ознаку;
- під час вибору врахувати силу зв'язку кожної ознаки з цільовою змінною `price`;
- вивести список сильно корельованих пар та перелік ознак, які видаляються.

Із тестової вибірки видалити ті самі ознаки.  
Не використовувати тестову вибірку для прийняття рішення про відбір ознак.


In [ ]:
num_features = cont_cols + ["symboling"]
corr = X_train[num_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Кореляція числових ознак (train)")
plt.show()

# зв'язок кожної ознаки з ціною теж тільки по train
corr_price = X_train[num_features].corrwith(y_train).abs()

pairs = []
for i, a in enumerate(num_features):
    for b in num_features[i + 1:]:
        if abs(corr.loc[a, b]) > 0.85:
            pairs.append((a, b, corr.loc[a, b]))
pairs.sort(key=lambda p: abs(p[2]), reverse=True)

print("Пари з |r| > 0.85:")
to_drop = []
for a, b, r in pairs:
    print(f"{a} - {b}: r = {r:.3f}  (з ціною: {a} {corr_price[a]:.3f}, {b} {corr_price[b]:.3f})")
    if a in to_drop or b in to_drop:
        continue
    # з пари залишаємо ту ознаку, яка сильніше пов'язана з ціною
    to_drop.append(a if corr_price[a] < corr_price[b] else b)

print()
print("Видаляємо ознаки:", to_drop)

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)
cont_cols = [c for c in cont_cols if c not in to_drop]

**Висновок:** знайшли 4 пари сильно корельованих ознак. citympg і highwaympg майже однакові (r = 0.96). З кожної пари залишили ознаку, яка сильніше пов'язана з ціною, тому видалили citympg, wheelbase, carlength і carwidth. З test видалили ті самі ознаки.

## 8. Підготовка числових і категоріальних ознак

Розділити ознаки на:
- неперервні числові;
- дискретну числову ознаку `symboling`;
- категоріальні.

Для неперервних числових ознак виконати послідовно:
1. перетворення Yeo–Johnson для зменшення асиметрії;
2. масштабування за допомогою StandardScaler.

Для `symboling` виконати масштабування без Yeo–Johnson.

Категоріальні ознаки закодувати методом One-Hot Encoding. Передбачити коректну обробку категорій у test, яких не було у train.

Усі перетворення об'єднати в єдиний механізм передобробки даних, який навчається тільки на `X_train`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder

cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print("Неперервні:", cont_cols)
print("Дискретна: ['symboling']")
print("Категоріальні:", cat_cols)

preprocess = ColumnTransformer([
    # спочатку Yeo-Johnson, потім стандартизація
    ("num", Pipeline([
        ("yeo_johnson", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", StandardScaler()),
    ]), cont_cols),
    ("symboling", StandardScaler(), ["symboling"]),
    # handle_unknown="ignore" - якщо в test буде нова категорія, помилки не буде
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

# категорії, яких немає в train
for col in cat_cols:
    new = set(X_test[col]) - set(X_train[col])
    if new:
        print(f"У test є нові категорії в {col}: {new}")

**Висновок:** вся передобробка зібрана в один ColumnTransformer: для неперервних ознак Yeo-Johnson і StandardScaler, для symboling тільки StandardScaler, для категоріальних One-Hot Encoding. У test є категорія three в cylindernumber, якої немає в train, тому в OneHotEncoder стоїть handle_unknown="ignore", і помилки не буде.

## 9. Перевірка трансформації числових ознак

Перевірити результат трансформації неперервних числових ознак:
- застосувати до train ту саму послідовність перетворень, яка буде використана в моделях;
- повторно обчислити коефіцієнти асиметрії;
- порівняти їх із початковими значеннями;
- зробити короткий висновок щодо ефективності трансформації.


In [ ]:
preprocess.fit(X_train)

# беремо тільки частину для неперервних ознак
num_part = preprocess.named_transformers_["num"].transform(X_train[cont_cols])
num_part = pd.DataFrame(num_part, columns=cont_cols)

skew_compare = pd.DataFrame({
    "До (вихідні дані)": df.loc[X_train.index, cont_cols].skew(),
    "Після обрізання викидів": X_train[cont_cols].skew(),
    "Після Yeo-Johnson": num_part.skew(),
})
skew_compare.round(3)

**Висновок:** після Yeo-Johnson асиметрія всіх неперервних ознак стала майже нульовою (за модулем не більше 0.07). Наприклад, у enginesize вона була 2.05, після обрізання викидів 0.96, а після трансформації 0.05. Тобто трансформація спрацювала добре, розподіли стали ближчими до нормального.

## 10. Побудова регресійних моделей

Побудувати та навчити такі моделі:
- Linear Regression;
- Ridge Regression;
- Lasso Regression;
- Random Forest Regressor;
- Gradient Boosting Regressor;
- Support Vector Regressor (SVR).

Для кожної моделі створити єдиний Pipeline, який містить:
1. попередню обробку ознак;
2. відповідну регресійну модель.

Використати однакову схему підготовки даних для коректного порівняння моделей.


In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# у SVR стандартний C=1 замалий, бо ціни - це тисячі
# підбираємо C крос-валідацією тільки на train
for C in [1, 1000, 10000, 50000, 100000]:
    svr = Pipeline([("prep", clone(preprocess)), ("model", SVR(C=C))])
    score = cross_val_score(svr, X_train, y_train, cv=5, scoring="r2").mean()
    print(f"SVR C={C}: R2 на крос-валідації = {score:.3f}")

З C=1 (за замовчуванням) у SVR R2 від'ємний, тобто модель гірша, ніж просто середнє значення. Це тому, що ціна вимірюється тисячами, а C=1 замалий. При C=50000 R2 на крос-валідації 0.878, а далі майже не росте, тому беремо C=50000. Підбирали тільки на train.

In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=10000),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(C=50000),
}

# для кожної моделі однакова передобробка + сама модель
models = {}
for name, reg in regressors.items():
    models[name] = Pipeline([("prep", clone(preprocess)), ("model", reg)])

print("Моделі:", list(models.keys()))
models["Random Forest"]

## 11. Оцінювання якості моделей

Для кожної моделі:
- виконати навчання на тренувальній вибірці;
- отримати прогноз для тестової вибірки;
- обчислити метрики **R², MAE, MSE та MAPE**.

`MAPE` подати у відсотках.

Результати всіх моделей об'єднати в одну порівняльну таблицю та впорядкувати за значенням R² від найбільшого до найменшого.

Під час інтерпретації врахувати:
- R² — чим більше, тим краще;
- MAE, MSE, MAPE — чим менше, тим краще.


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Модель": name,
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "MAPE, %": mean_absolute_percentage_error(y_test, y_pred) * 100,
        "R2 на train": model.score(X_train, y_train),
    })

results = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results["MSE"] = results["MSE"].round().astype(int)  # щоб MSE не показувався як e+06
results.round(4)

**Висновок:** найкращий результат у Random Forest: R2 = 0.954, MAE близько 1351, MAPE близько 9.9%. Далі SVR і Gradient Boosting (R2 близько 0.93), лінійні моделі гірші (R2 0.85-0.88, MAPE 15-18%). На train у всіх моделей R2 вищий, ніж на test, тобто є невелике перенавчання, найменше воно у Random Forest.

## 12. Пояснення вибору типу моделі

Пояснити, чому класифікаційна модель, наприклад Logistic Regression, не підходить для безпосереднього прогнозування неперервної ціни автомобіля.

Зазначити, до якого типу задачі належить прогнозування `price` та чому моделі, використані в роботі, відповідають цій задачі.


**Відповідь:** прогнозування ціни це задача регресії, бо ціна це неперервне число (наприклад 13495 або 41315), а не клас.

Logistic Regression це модель класифікації. Вона видає ймовірність належності до класу (від 0 до 1) і відповідь типу так/ні, тому напряму передбачити ціну нею не можна. Можна було б поділити ціни на групи (дешеві, середні, дорогі), але тоді ми втратимо точне значення ціни.

Всі моделі в роботі (Linear Regression, Ridge, Lasso, Random Forest Regressor, Gradient Boosting Regressor, SVR) регресійні, вони на виході дають число, тому підходять для цієї задачі.

## 13. Візуальне порівняння прогнозів

Для кожної побудованої моделі зобразити співвідношення:
- фактичних значень `price`;
- прогнозованих значень `price`.

Додати лінію ідеального прогнозу та проаналізувати, наскільки близько до неї розташовані прогнозовані значення. Звернути увагу на великі відхилення.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[i]
    ax.scatter(y_test, y_pred, alpha=0.7)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")  # ідеальний прогноз
    ax.set_title(f"{name}, R2 = {r2_score(y_test, y_pred):.3f}")
    ax.set_xlabel("Фактична ціна")
    ax.set_ylabel("Прогнозована ціна")

plt.tight_layout()
plt.show()

# машини з найбільшою помилкою у найкращої моделі
best = results.loc[0, "Модель"]
errors = pd.DataFrame({"car_ID": id_test, "Фактична ціна": y_test, "Прогноз": predictions[best]})
errors["Помилка"] = (errors["Фактична ціна"] - errors["Прогноз"]).abs()
print("Найбільші помилки моделі", best)
errors.sort_values("Помилка", ascending=False).head(5).round(0)

**Висновок:** у Random Forest, SVR і Gradient Boosting точки лежать ближче до червоної лінії, у лінійних моделей розкид більший. Найбільші помилки у дорогих машин, бо таких машин у даних мало. Наприклад, машину з car_ID 17 за 41315 Random Forest оцінив приблизно в 36000, а машину з car_ID 16 за 30760 навпаки приблизно в 36000.

## 14. Прогноз для 10 автомобілів

Випадково вибрати 10 автомобілів із тестової вибірки.

За допомогою моделі Random Forest отримати для них прогноз ціни та сформувати таблицю з такими даними:
- `car_ID`;
- фактична ціна;
- прогнозована ціна;
- абсолютна помилка;
- абсолютна відсоткова помилка.

Прокоментувати отримані результати.


In [ ]:
sample = X_test.sample(10, random_state=42)
rf_pred = models["Random Forest"].predict(sample)

table = pd.DataFrame({
    "car_ID": id_test.loc[sample.index].values,
    "Фактична ціна": y_test.loc[sample.index].values,
    "Прогнозована ціна": rf_pred,
})
table["Абсолютна помилка"] = (table["Фактична ціна"] - table["Прогнозована ціна"]).abs()
table["Абсолютна відсоткова помилка, %"] = table["Абсолютна помилка"] / table["Фактична ціна"] * 100

print("Середня відсоткова помилка на цих 10 машинах:", round(table["Абсолютна відсоткова помилка, %"].mean(), 2), "%")
table.round(2)

**Висновок:** на 10 випадкових машинах середня помилка Random Forest 8.73%. Для більшості машин помилка менше 10%, найгірше з машиною car_ID 195: модель дала 16647 замість 12940 (помилка 28.65%). У найдорожчої машини (car_ID 17) абсолютна помилка найбільша, понад 5000, але у відсотках це 12.75%.

## 15. Висновки

Сформулювати підсумкові висновки за результатами роботи.

У висновках:
- порівняти якість усіх моделей;
- проаналізувати R², MAE, MSE і MAPE;
- зазначити, яка модель показала найкращі результати за отриманими метриками;
- прокоментувати величину похибок;
- звернути увагу на можливі ознаки перенавчання;
- коротко оцінити вплив попередньої обробки даних на результати моделювання.


**Висновки**

1. Побудували 6 моделей регресії з однаковою передобробкою даних. Найкраще себе показав Random Forest: R2 = 0.954, MAE близько 1351, MSE близько 3.6 млн, MAPE 9.86%.
2. На другому і третьому місці SVR (R2 0.932, але тільки після підбору C) і Gradient Boosting (R2 0.930). Лінійні моделі слабші: у Linear Regression і Lasso R2 близько 0.88, у Ridge 0.854, а помилка 15-18%.
3. MSE виходить у мільйонах, бо помилки в доларах підносяться до квадрату, тому зручніше дивитись на MAE і MAPE. Помилка близько 1350 при середній ціні машини близько 13 тисяч це непогана точність.
4. Є ознаки перенавчання: на train R2 у всіх моделей вищий, ніж на test. Найбільша різниця у Ridge і Linear Regression (0.08-0.09), у Gradient Boosting і SVR близько 0.06, найменша у Random Forest (0.03). Даних всього 205, тому моделі легко підлаштовуються під train.
5. Передобробка дала помітний ефект: обрізання викидів і Yeo-Johnson прибрали асиметрію, видалення корельованих ознак прибрало дублювання інформації, а замість 147 назв моделей ми використали 22 марки. Без підбору параметра C модель SVR взагалі не працювала (R2 менше нуля).

## Загальні вимоги до виконання

- Не використовувати тестову вибірку для визначення меж викидів, кореляційного відбору ознак, навчання трансформерів, масштабування або кодування категорій.
- Усі параметри передобробки мають визначатися тільки за навчальною вибіркою.
- Фінальні метрики моделей обчислювати на тестовій вибірці.
- Не використовувати `car_ID` як предиктор.
- Не видаляти спостереження або ознаки без пояснення причини.
- До кожного основного етапу додати короткий змістовний висновок.
- У підсумкових висновках спиратися на отримані метрики та графіки.
